In [0]:
-- ============================================================
-- GOLD MART: readmissions by age band
-- Source for the readmission age-gradient bar chart.
-- ============================================================
CREATE OR REPLACE TABLE health_lakehouse.gold.mart_readmissions_by_ageband AS
WITH inpatient_stays AS (
  SELECT patient_id, start_ts AS admit_ts, stop_ts AS discharge_ts
  FROM health_lakehouse.gold.fact_encounters
  WHERE encounter_class = 'inpatient' AND stop_ts IS NOT NULL
),
sequenced AS (
  SELECT *, LEAD(admit_ts) OVER (PARTITION BY patient_id ORDER BY admit_ts) AS next_admit_ts
  FROM inpatient_stays
),
flagged AS (
  SELECT *,
    CASE WHEN next_admit_ts IS NOT NULL
          AND DATEDIFF(next_admit_ts, discharge_ts) BETWEEN 0 AND 30
         THEN 1 ELSE 0 END AS is_readmission_30d
  FROM sequenced
)
SELECT
  p.age_band,
  COUNT(*) AS discharges,
  SUM(f.is_readmission_30d) AS readmissions,
  ROUND(100.0 * SUM(f.is_readmission_30d) / COUNT(*), 2) AS readmission_rate_pct
FROM flagged f
JOIN health_lakehouse.gold.dim_patient p ON f.patient_id = p.patient_id
GROUP BY p.age_band
ORDER BY p.age_band;

In [0]:
-- ============================================================
-- GOLD MART: comorbidity lift (top associated disorder pairs)
-- Source for the comorbidity ranked-bar / heatmap.
-- ============================================================
CREATE OR REPLACE TABLE health_lakehouse.gold.mart_comorbidity_lift AS
WITH patient_disorders AS (
  SELECT DISTINCT patient_id, condition_code, condition_desc
  FROM health_lakehouse.gold.fact_conditions
  WHERE condition_type = 'disorder'
),
total_patients AS (SELECT COUNT(DISTINCT patient_id) AS n_total FROM patient_disorders),
prevalence AS (
  SELECT condition_code, condition_desc, COUNT(DISTINCT patient_id) AS n_with
  FROM patient_disorders GROUP BY condition_code, condition_desc
),
pairs AS (
  SELECT a.condition_code AS code_a, b.condition_code AS code_b,
         COUNT(DISTINCT a.patient_id) AS n_both
  FROM patient_disorders a
  JOIN patient_disorders b
    ON a.patient_id = b.patient_id AND a.condition_code < b.condition_code
  GROUP BY a.condition_code, b.condition_code
)
SELECT
  pa.condition_desc AS condition_a,
  pb.condition_desc AS condition_b,
  p.n_both,
  ROUND((p.n_both * 1.0 * t.n_total) / (pa.n_with * pb.n_with), 2) AS lift
FROM pairs p
JOIN prevalence pa ON p.code_a = pa.condition_code
JOIN prevalence pb ON p.code_b = pb.condition_code
CROSS JOIN total_patients t
WHERE p.n_both >= 30
ORDER BY lift DESC
LIMIT 20;

In [0]:
-- ============================================================
-- GOLD MART: hypertension care cascade funnel
-- Source for the funnel chart.
-- ============================================================
CREATE OR REPLACE TABLE health_lakehouse.gold.mart_care_cascade AS
WITH diagnosed AS (
  SELECT patient_id, MIN(onset_date) AS dx_date
  FROM health_lakehouse.gold.fact_conditions
  WHERE condition_code = '59621000' GROUP BY patient_id
),
followed_up AS (
  SELECT DISTINCT d.patient_id, d.dx_date
  FROM diagnosed d
  JOIN health_lakehouse.gold.fact_encounters e
    ON e.patient_id = d.patient_id
   AND e.encounter_date > d.dx_date
   AND e.encounter_date <= d.dx_date + INTERVAL 90 DAYS
),
medicated AS (
  SELECT DISTINCT f.patient_id, f.dx_date, MIN(m.start_date) AS first_med_date
  FROM followed_up f
  JOIN health_lakehouse.gold.fact_medications m
    ON m.patient_id = f.patient_id AND m.start_date >= f.dx_date
   AND LOWER(m.medication_desc) RLIKE
       'lisinopril|hydrochlorothiazide|amlodipine|losartan|metoprolol|furosemide|valsartan|enalapril|atenolol|benazepril|bisoprolol|olmesartan|sacubitril'
  GROUP BY f.patient_id, f.dx_date
),
controlled AS (
  SELECT DISTINCT mo.patient_id
  FROM medicated mo
  JOIN health_lakehouse.gold.fact_observations o
    ON o.patient_id = mo.patient_id AND o.measure_group = 'vital: systolic_bp'
   AND o.observed_ts >= mo.first_med_date AND o.value_numeric < 140
)
SELECT stage_order, stage, patients,
       ROUND(100.0 * patients / FIRST(patients) OVER (ORDER BY stage_order), 1) AS pct_of_diagnosed
FROM (
  SELECT 1 AS stage_order, '1. Diagnosed' AS stage, COUNT(*) AS patients FROM diagnosed
  UNION ALL SELECT 2, '2. Followed up (=90d)', COUNT(*) FROM followed_up
  UNION ALL SELECT 3, '3. Medicated', COUNT(*) FROM medicated
  UNION ALL SELECT 4, '4. Controlled (<140)', COUNT(*) FROM controlled
)
ORDER BY stage_order;

In [0]:
-- ============================================================
-- GOLD MART: CKD progression pathways
-- Source for the progression funnel / pathway chart.
-- ============================================================
CREATE OR REPLACE TABLE health_lakehouse.gold.mart_ckd_pathways AS
WITH stage_events AS (
  SELECT patient_id, ckd_stage, MIN(onset_date) AS stage_onset
  FROM health_lakehouse.gold.fact_conditions
  WHERE ckd_stage IS NOT NULL
  GROUP BY patient_id, ckd_stage
),
journeys AS (
  SELECT
    patient_id,
    array_join(
      transform(
        array_sort(collect_list(struct(stage_onset, ckd_stage))),
        x -> CAST(x.ckd_stage AS STRING)
      ), ' → '
    ) AS pathway,
    MIN(stage_onset) AS journey_start,
    MAX(stage_onset) AS journey_end
  FROM stage_events
  GROUP BY patient_id
)
SELECT
  pathway,
  COUNT(*) AS patients,
  ROUND(AVG(DATEDIFF(journey_end, journey_start) / 365.25), 1) AS avg_journey_years
FROM journeys
GROUP BY pathway
ORDER BY patients DESC
LIMIT 15;

In [0]:
-- ============================================================
-- GOLD MART: CKD stage transitions (for a funnel/flow viz)
-- Forward transitions only; transplant reversals excluded.
-- ============================================================
CREATE OR REPLACE TABLE health_lakehouse.gold.mart_ckd_transitions AS
WITH stage_events AS (
  SELECT patient_id, ckd_stage, MIN(onset_date) AS stage_onset
  FROM health_lakehouse.gold.fact_conditions
  WHERE ckd_stage IS NOT NULL
  GROUP BY patient_id, ckd_stage
),
sequenced AS (
  SELECT patient_id, ckd_stage, stage_onset,
         LEAD(ckd_stage)   OVER (PARTITION BY patient_id ORDER BY stage_onset) AS next_stage,
         LEAD(stage_onset) OVER (PARTITION BY patient_id ORDER BY stage_onset) AS next_onset
  FROM stage_events
)
SELECT
  ckd_stage AS from_stage,
  next_stage AS to_stage,
  COUNT(*) AS transitions,
  ROUND(AVG(DATEDIFF(next_onset, stage_onset) / 365.25), 1) AS avg_years_in_stage
FROM sequenced
WHERE next_stage IS NOT NULL
  AND next_stage > ckd_stage -- forward only
GROUP BY ckd_stage, next_stage
ORDER BY from_stage, to_stage;

In [0]:
-- ============================================================
-- GOLD MART: CKD cohort cost by age band
-- Source for the age-stratified cost comparison chart.
-- ============================================================
CREATE OR REPLACE TABLE health_lakehouse.gold.mart_cohort_cost_by_ageband AS
WITH ckd_cohort AS (
  SELECT DISTINCT patient_id
  FROM health_lakehouse.gold.fact_conditions
  WHERE ckd_stage IS NOT NULL
),
patient_costs AS (
  SELECT
    p.age_band,
    CASE WHEN k.patient_id IS NOT NULL THEN 'CKD' ELSE 'non_CKD' END AS cohort,
    p.healthcare_expenses AS lifetime_expenses
  FROM health_lakehouse.gold.dim_patient p
  LEFT JOIN ckd_cohort k ON p.patient_id = k.patient_id
)
SELECT
  age_band,
  ROUND(AVG(CASE WHEN cohort = 'CKD' THEN lifetime_expenses END), 0) AS ckd_avg,
  ROUND(AVG(CASE WHEN cohort = 'non_CKD' THEN lifetime_expenses END), 0) AS non_ckd_avg,
  SUM(CASE WHEN cohort = 'CKD' THEN 1 ELSE 0 END) AS ckd_n,
  SUM(CASE WHEN cohort = 'non_CKD' THEN 1 ELSE 0 END) AS non_ckd_n,
  ROUND(
    AVG(CASE WHEN cohort = 'CKD' THEN lifetime_expenses END) /
    NULLIF(AVG(CASE WHEN cohort = 'non_CKD' THEN lifetime_expenses END), 0), 2
  ) AS cost_ratio
FROM patient_costs
GROUP BY age_band
ORDER BY age_band;

In [0]:
-- ============================================================
-- GOLD MART: medication persistence by drug class
-- ============================================================
CREATE OR REPLACE TABLE health_lakehouse.gold.mart_med_persistence AS
WITH chronic_fills AS (
  SELECT patient_id,
    CASE
      WHEN LOWER(medication_desc) RLIKE 'lisinopril|enalapril|benazepril' THEN 'ACE inhibitor'
      WHEN LOWER(medication_desc) RLIKE 'losartan|valsartan|olmesartan' THEN 'ARB'
      WHEN LOWER(medication_desc) RLIKE 'amlodipine' THEN 'Calcium channel blocker'
      WHEN LOWER(medication_desc) RLIKE 'hydrochlorothiazide|furosemide' THEN 'Diuretic'
      WHEN LOWER(medication_desc) RLIKE 'metoprolol|atenolol|bisoprolol' THEN 'Beta blocker'
      WHEN LOWER(medication_desc) RLIKE 'metformin|glipizide' THEN 'Oral antidiabetic'
      WHEN LOWER(medication_desc) RLIKE 'insulin' THEN 'Insulin'
      WHEN LOWER(medication_desc) RLIKE 'statin|simvastatin|atorvastatin' THEN 'Statin'
      ELSE 'Other' END AS drug_class,
    LEAST(duration_days, 730) AS covered_days
  FROM health_lakehouse.gold.fact_medications
  WHERE duration_days IS NOT NULL AND duration_days > 0
),
persistence AS (
  SELECT patient_id, drug_class, COUNT(*) AS n_fills,
         ROUND(SUM(covered_days)/365.25, 1) AS years_covered
  FROM chronic_fills WHERE drug_class <> 'Other'
  GROUP BY patient_id, drug_class HAVING COUNT(*) >= 3
)
SELECT drug_class,
       COUNT(DISTINCT patient_id) AS patients,
       ROUND(AVG(n_fills), 1) AS avg_fills,
       ROUND(AVG(years_covered), 1) AS avg_years_covered
FROM persistence GROUP BY drug_class ORDER BY patients DESC;

In [0]:
SHOW TABLES IN health_lakehouse.gold LIKE 'mart_*';